# 01 - Ingesta ESIOS -> raw

Este notebook inventaría el catálogo ESIOS y descarga, sin transformar, el conjunto de indicadores seleccionado para el primer modelo. Los datos se guardan localmente en `data/raw/esios/`. La subida a S3 está desactivada por defecto y solo se activa explícitamente si se dispone de credenciales AWS.

El periodo objetivo del modelo es `2025-10-01` a `2026-09-30`. La ventana de ingesta empieza en `2024-10-01` para poder calcular los retardos de aproximadamente un año (`lag_1y`) desde el primer intervalo objetivo. REData debe conservar la misma ventana histórica ampliada.

La lista de indicadores se define en la celda de configuración e incluye demanda, generación, programas de intercambio, capacidad de interconexión, hidráulica y reservas.

**Importante:** revisa primero el inventario y la estimación de peticiones. La descarga solo se inicia al mantener `RUN_FULL_DOWNLOAD` en `True`. La API puede ofrecer datos vacíos para ciertos indicadores o periodos; esos JSON se conservan igualmente. La ingesta no demuestra por sí sola que un pronóstico o programa tenga un snapshot histórico as-of: para el backtest habrá que conservar la versión disponible antes de cada `forecast_origin`.

In [1]:
import csv
import html
import json
import os
import re
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto (requirements.txt y notebooks/).")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("ESIOS_API_KEY")
BUCKET = os.getenv("S3_BUCKET")
UPLOAD_TO_S3 = False
BASE_URL = "https://api.esios.ree.es"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "esios"

if not API_KEY:
    raise RuntimeError(f"Falta ESIOS_API_KEY en {PROJECT_ROOT / '.env'}.")
if UPLOAD_TO_S3 and not BUCKET:
    raise RuntimeError("UPLOAD_TO_S3 está activado, pero falta S3_BUCKET en el archivo .env.")

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Destino local: {RAW_DIR}")
print(f"Subida S3: {'activada' if UPLOAD_TO_S3 else 'desactivada'}")

Raíz del proyecto: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer
Destino local: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer\data\raw\esios
Subida S3: desactivada


In [ ]:
# Ventana de ingesta: un año adicional para calcular lag_1y del periodo objetivo.
# El periodo objetivo del modelo sigue siendo 2025-10-01 a 2026-09-30.
START_DATE = date(2024, 10, 1)
END_DATE = date(2026, 9, 30)

# Primer conjunto de indicadores para el modelo.
# Los pronósticos/programas solo son válidos como variables futuras si se
# dispone de un snapshot publicado antes de forecast_origin.
INDICATOR_IDS: list[int] | None = [
    # Programas PBF por frontera: Francia, Portugal, Marruecos y Andorra.
    28, 29, 30, 31, 32, 33, 34, 35,
    # Demanda, indisponibilidad y potencia disponible.
    460, 462, 472, 473, 474, 475, 476, 477, 478, 479,
    # Capacidad ATC Francia y previsiones NTC de Francia, Portugal y Marruecos.
    506, 507, 1844, 1845, 1846, 1848, 1849, 1850,
    # Previsiones eólicas, solares y demanda/programación.
    541, 542, 543, 545, 603,
    # Generación observada por tecnología.
    546, 547, 549, 551, 552, 1297, 2041, 2051,
    # Capacidad Baleares, llenado hidráulico y reserva.
    570, 623, 628,
]

# Controles para la descarga.
CHUNK_DAYS = 30
PAUSE_SECONDS = 1
MAX_WORKERS = 3
RETRIES = 4
FORCE = False
RUN_FULL_DOWNLOAD = True

if END_DATE < START_DATE:
    raise ValueError("END_DATE debe ser igual o posterior a START_DATE.")

print(f"Ventana de ingesta: {START_DATE} a {END_DATE}")
print("Periodo objetivo del modelo: 2025-10-01 a 2026-09-30")
print(f"Indicadores seleccionados: {len(INDICATOR_IDS)}")
print(f"Descargas simultáneas máximas: {MAX_WORKERS}")

Ventana de ingesta: 2024-10-01 a 2026-09-30
Periodo objetivo del modelo: 2025-10-01 a 2026-09-30
Indicadores seleccionados: 42


In [3]:
HEADERS = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "x-api-key": API_KEY,
}


def get_json(url: str, params: dict | None = None) -> dict:
    for attempt in range(RETRIES):
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=90)
            if response.status_code in (401, 403):
                raise PermissionError(
                    f"ESIOS rechazó la credencial o el acceso (HTTP {response.status_code}). "
                    "Revisa ESIOS_API_KEY y sus permisos."
                )
            if response.status_code == 429 or response.status_code >= 500:
                response.raise_for_status()
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict):
                raise ValueError("ESIOS devolvió una respuesta JSON que no es un objeto.")
            return payload
        except requests.RequestException as exc:
            if attempt == RETRIES - 1:
                raise RuntimeError(f"Falló la petición a ESIOS: {url}") from exc
            wait = 2 ** attempt
            print(f"Petición fallida; reintento {attempt + 1}/{RETRIES} en {wait}s: {exc}")
            time.sleep(wait)


def save_raw(payload: dict, path: Path, s3_key: str | None = None) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    if UPLOAD_TO_S3 and s3_key:
        import boto3

        boto3.client("s3").upload_file(str(path), BUCKET, s3_key)
        print(f"  subido a s3://{BUCKET}/{s3_key}")
    return path


def date_chunks(start: date, end: date, size: int = CHUNK_DAYS):
    current = start
    while current <= end:
        chunk_end = min(current + timedelta(days=size - 1), end)
        yield current, chunk_end
        current = chunk_end + timedelta(days=1)


def values_count(payload: dict) -> int:
    indicator = payload.get("indicator")
    if not isinstance(indicator, dict) or not isinstance(indicator.get("values"), list):
        raise ValueError("Respuesta de indicador no válida: se esperaba indicator.values como lista.")
    return len(indicator["values"])

## 1. Inventario del catálogo

Esta celda consulta y guarda la respuesta del catálogo sin modificarla. El número de indicadores y la estimación de llamadas dependen de lo que ESIOS publique al ejecutar el notebook.

In [4]:
catalog_payload = get_json(f"{BASE_URL}/indicators", params={"locale": "es"})
indicators = catalog_payload.get("indicators")
if not isinstance(indicators, list):
    raise ValueError("Respuesta del catálogo no válida: falta la lista 'indicators'.")

downloaded_at = datetime.now(timezone.utc)
catalog_name = f"catalogo_indicadores_{downloaded_at.strftime('%Y%m%dT%H%M%SZ')}.json"
catalog_path = RAW_DIR / "catalogo" / catalog_name
save_raw(catalog_payload, catalog_path, f"raw/esios/catalogo/{catalog_name}")

def clean_description(value: str) -> str:
    text = re.sub(r"<[^>]+>", " ", value or "")
    return " ".join(html.unescape(text).split())

catalog_dir = PROJECT_ROOT / "docs" / "fuentes" / "esios"
catalog_dir.mkdir(parents=True, exist_ok=True)
catalog_csv = catalog_dir / "catalogo_indicadores_esios.csv"
publication_label = "Publicación:"
with catalog_csv.open("w", encoding="utf-8-sig", newline="") as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=["id", "name", "short_name", "description", "publication"],
    )
    writer.writeheader()
    for item in indicators:
        if not isinstance(item, dict) or item.get("id") is None:
            continue
        description, has_publication, publication = clean_description(
            item.get("description", "")
        ).partition(publication_label)
        writer.writerow({
            "id": item["id"],
            "name": item.get("name", ""),
            "short_name": item.get("short_name", ""),
            "description": description.strip(),
            "publication": publication.strip() if has_publication else "",
        })

catalog_doc = catalog_dir / "catalogo_indicadores_esios.md"
catalog_doc.write_text(
    "# Catálogo de indicadores ESIOS\n\n"
    f"Snapshot obtenido el {downloaded_at.date()} ({len(indicators)} indicadores).\n\n"
    "El [CSV completo](./catalogo_indicadores_esios.csv) incluye el ID, nombre, nombre corto, "
    "descripción y publicación indicada por ESIOS. La descripción se ha limpiado de etiquetas "
    "HTML; la publicación es una columna separada para facilitar la lectura y el filtrado. "
    "Estos metadatos no validan por sí solos la cobertura ni la idoneidad de cada indicador.\n\n"
    f"La respuesta original del catálogo se conserva en `{catalog_path.relative_to(PROJECT_ROOT)}`. "
    "Fuente: [API ESIOS](https://api.esios.ree.es/indicators).\n",
    encoding="utf-8",
)

catalog_ids_all = sorted({
    int(item["id"])
    for item in indicators
    if isinstance(item, dict) and item.get("id") is not None
})
if INDICATOR_IDS is None:
    catalog_ids = catalog_ids_all
else:
    unknown_ids = sorted(set(INDICATOR_IDS) - set(catalog_ids_all))
    if unknown_ids:
        raise ValueError(f"IDs no encontrados en el catálogo ESIOS: {unknown_ids}")
    catalog_ids = sorted(set(INDICATOR_IDS))
n_chunks = sum(1 for _ in date_chunks(START_DATE, END_DATE))
estimated_requests = len(catalog_ids) * n_chunks

print(f"Indicadores disponibles en catálogo: {len(catalog_ids_all)}")
print(f"Indicadores seleccionados para ingesta: {len(catalog_ids)} {catalog_ids}")
print(f"Bloques por indicador: {n_chunks}")
print(f"Peticiones estimadas: {estimated_requests:,}")
print(f"Inventario original guardado en: {catalog_path}")
print(f"CSV del catálogo guardado en: {catalog_csv}")

Indicadores disponibles en catálogo: 1982
Indicadores seleccionados para ingesta: 42 [28, 29, 30, 31, 32, 33, 34, 35, 460, 462, 472, 473, 474, 475, 476, 477, 478, 479, 506, 507, 541, 542, 543, 545, 546, 547, 549, 551, 552, 570, 603, 623, 628, 1297, 1844, 1845, 1846, 1848, 1849, 1850, 2041, 2051]
Bloques por indicador: 25
Peticiones estimadas: 1,050
Inventario original guardado en: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer\data\raw\esios\catalogo\catalogo_indicadores_20261112T103609Z.json
CSV del catálogo guardado en: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer\docs\fuentes\esios\catalogo_indicadores_esios.csv


## 2. Descarga de todas las series

Revisa la estimación anterior antes de continuar. Para lanzar la descarga, cambia `RUN_FULL_DOWNLOAD` a `True` en la celda de configuración y vuelve a ejecutar las celdas desde arriba. Los bloques guardados se omiten, por lo que se puede reanudar; usa `FORCE = True` para volver a descargarlos.

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def download_one(task):
    indicator_id, chunk_start, chunk_end = task
    filename = f"indicador_{indicator_id}_{chunk_start}_{chunk_end}.json"
    local_path = RAW_DIR / "indicadores" / str(indicator_id) / filename
    s3_key = f"raw/esios/indicadores/{indicator_id}/{filename}"

    if local_path.exists() and not FORCE:
        return "skipped", indicator_id, chunk_start, chunk_end, 0, None

    url = f"{BASE_URL}/indicators/{indicator_id}"
    params = {
        "start_date": f"{chunk_start.isoformat()}T00:00:00Z",
        "end_date": f"{chunk_end.isoformat()}T23:59:59Z",
        "locale": "es",
    }
    try:
        payload = get_json(url, params=params)
        n_values = values_count(payload)
        save_raw(payload, local_path, s3_key)
        time.sleep(PAUSE_SECONDS)
        return "downloaded", indicator_id, chunk_start, chunk_end, n_values, None
    except (RuntimeError, PermissionError, ValueError) as exc:
        return "failed", indicator_id, chunk_start, chunk_end, 0, str(exc)


if not RUN_FULL_DOWNLOAD:
    print("Descarga masiva desactivada. Revisa la estimación y cambia RUN_FULL_DOWNLOAD a True para iniciar.")
else:
    downloaded = 0
    skipped = 0
    empty = 0
    failures = []
    chunks = list(date_chunks(START_DATE, END_DATE))
    tasks = [
        (indicator_id, chunk_start, chunk_end)
        for indicator_id in catalog_ids
        for chunk_start, chunk_end in chunks
    ]

    print(f"Tareas de descarga: {len(tasks)}")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(download_one, task) for task in tasks]
        for future in as_completed(futures):
            status, indicator_id, chunk_start, chunk_end, n_values, error = future.result()
            if status == "skipped":
                skipped += 1
                continue
            if status == "downloaded":
                downloaded += 1
                empty += n_values == 0
                print(f"Indicador {indicator_id} | {chunk_start}..{chunk_end} | {n_values} valores")
                continue
            failures.append((indicator_id, chunk_start, chunk_end, error))
            print(f"ERROR indicador {indicator_id} | {chunk_start}..{chunk_end}: {error}")

    print(
        f"\nResumen: descargados={downloaded}, ya existentes={skipped}, "
        f"bloques vacíos={empty}, fallidos={len(failures)}"
    )
    if failures:
        failures_path = RAW_DIR / "fallos_ingesta.json"
        failures_path.parent.mkdir(parents=True, exist_ok=True)
        failures_path.write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")
        raise RuntimeError(f"Hay bloques fallidos; detalle guardado en {failures_path}.")

Indicador 28 | 2024-10-01..2024-10-30 | 451 valores
Indicador 28 | 2024-10-31..2024-11-29 | 477 valores
Indicador 28 | 2024-11-30..2024-12-29 | 517 valores
Indicador 28 | 2024-12-30..2025-01-28 | 326 valores
Indicador 28 | 2025-01-29..2025-02-27 | 179 valores
Indicador 28 | 2025-02-28..2025-03-29 | 95 valores
Indicador 28 | 2025-03-30..2025-04-28 | 149 valores
Indicador 28 | 2025-04-29..2025-05-28 | 208 valores
Indicador 28 | 2025-05-29..2025-06-27 | 606 valores
Indicador 28 | 2025-06-28..2025-07-27 | 530 valores
Indicador 28 | 2025-07-28..2025-08-26 | 528 valores
Indicador 28 | 2025-08-27..2025-09-25 | 547 valores
Indicador 28 | 2025-09-26..2025-10-25 | 1763 valores
Indicador 28 | 2025-10-26..2025-11-24 | 1754 valores
Indicador 28 | 2025-11-25..2025-12-24 | 2012 valores
Indicador 28 | 2025-12-25..2026-01-23 | 1242 valores
Indicador 28 | 2026-01-24..2026-02-22 | 174 valores
Indicador 28 | 2026-02-23..2026-03-24 | 990 valores
Indicador 28 | 2026-03-25..2026-04-23 | 1103 valores
Indicado